# NOAA Environmental Data & Thermal Image Correlation Analysis

**Site:** Richard Weeks Hall  
**Coordinates:** LAT = 40.52444606321408, LON = -74.45961334547036

This notebook:
1. Fetches NOAA hourly environmental data (CDO global-hourly dataset)
2. Processes uploaded thermal images and extracts intensity metrics
3. Correlates image metrics with environmental parameters

## Setup & Dependencies

In [1]:
!pip install requests pandas


In [2]:
import requests
import pandas as pd


In [9]:
LAT = 40.52444606321408
LON = -74.45961334547036
DATE = "2025-03-19"
SITE_LABEL = "Richard Weeks Hall"


In [10]:
points_url = f"https://api.weather.gov/points/{LAT},{LON}"
headers = {"User-Agent": "Rutgers-Research"}

resp = requests.get(points_url, headers=headers)
resp.raise_for_status()

stations_url = resp.json()["properties"]["observationStations"]
stations = requests.get(stations_url, headers=headers).json()["features"]

station_id = stations[0]["properties"]["stationIdentifier"]
print("Using NWS Station:", station_id)


Using NWS Station: KSMQ


In [15]:
obs_url = f"https://api.weather.gov/stations/{station_id}/observations"

params = {
    "start": f"{DATE}T00:00:00Z",
    "end": f"{DATE}T23:59:59Z",
    "limit": 500
}

resp = requests.get(
    obs_url,
    headers={
        "User-Agent": "Rutgers-Research",
        "Accept": "application/geo+json"
    },
    params=params
)

resp.raise_for_status()
features = resp.json().get("features", [])

print(f"Number of observations returned: {len(features)}")


Number of observations returned: 0


In [16]:
DATE = "2025-03-19"
STATION = "KEWR"  # Newark Liberty (closest major METAR to RWH)


In [17]:
url = "https://aviationweather.gov/api/data/metar"

params = {
    "ids": STATION,
    "format": "json",
    "start": f"{DATE}T00:00:00Z",
    "end": f"{DATE}T23:59:59Z"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()
print("Number of observations:", len(data))


Number of observations: 1


In [18]:
list(data[0].keys())


['icaoId',
 'receiptTime',
 'obsTime',
 'reportTime',
 'temp',
 'dewp',
 'wdir',
 'wspd',
 'visib',
 'altim',
 'slp',
 'qcField',
 'metarType',
 'rawOb',
 'lat',
 'lon',
 'elev',
 'name',
 'cover',
 'clouds',
 'fltCat']

In [20]:
df_clean = pd.DataFrame({
    "timestamp_utc": pd.to_datetime(df_hourly["timestamp_utc"], unit="s", utc=True),
    "temperature_C": df_hourly["temperature_C"],
    "dewpoint_C": df_hourly["dewpoint_C"],
    "wind_speed_mps": df_hourly["wind_speed_mps"],
    "wind_direction_deg": df_hourly["wind_direction_deg"],
    "pressure_hPa": df_hourly["pressure_hPa"],
    "visibility_miles": df_hourly["visibility_miles"],
    "raw_metar": df_hourly["raw_metar"]
})

df_clean


,timestamp_utc,temperature_C,dewpoint_C,wind_speed_mps,wind_direction_deg,pressure_hPa,visibility_miles,raw_metar
0,2026-02-09 22:51:00+00:00,0,-15.6,6,320,1022.8,10+,METAR KEWR 092251Z 32006KT 10SM FEW100 SCT180 ...


In [ ]:
import os
import re
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image
from PIL.ExifTags import TAGS
import pvlib
from pvlib.solarposition import get_solarposition
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings('ignore')
print("All dependencies loaded successfully.")

All dependencies loaded successfully.


## Configuration Cell

**Edit the values below before running:**

In [ ]:
# =============================================================================
# USER CONFIGURATION - EDIT THESE VALUES
# =============================================================================

# Your NOAA CDO API token (get one free at: https://www.ncdc.noaa.gov/cdo-web/token)
NOAA_API_KEY = "e97d47ec-be0b-11ee-84d9-0242ac130002-e97d48aa-be0b-11ee-84d9-0242ac130002"

# Time window for data retrieval (UTC, ISO format)
START_UTC = "2024-01-01T00:00:00"
END_UTC = "2024-01-07T23:59:59"

# Station ID (leave empty string to auto-select nearest station)
STATION_ID = ""

# =============================================================================
# FIXED SITE PARAMETERS - DO NOT MODIFY
# =============================================================================
LAT = 40.52444606321408
LON = -74.45961334547036
SITE_LABEL = "Richard Weeks Hall"

# Output paths
OUTPUT_DIR = "/content"
THERMAL_DIR = "/content/thermal"
os.makedirs(THERMAL_DIR, exist_ok=True)

print(f"Site: {SITE_LABEL}")
print(f"Coordinates: ({LAT}, {LON})")
print(f"Time window: {START_UTC} to {END_UTC}")

Site: Richard Weeks Hall
Coordinates: (40.52444606321408, -74.45961334547036)
Time window: 2024-01-01T00:00:00 to 2024-01-07T23:59:59


---
# Phase A: NOAA Data Retrieval

## NOAA CDO API Functions

In [ ]:
BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"

def noaa_request(endpoint, params, api_key, max_retries=5):
    """Make a request to NOAA CDO API with retry logic for rate limits."""
    headers = {"token": api_key}
    url = f"{BASE_URL}/{endpoint}"

    for attempt in range(max_retries):
        response = requests.get(url, headers=headers, params=params)

        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait_time = 2 ** attempt
            print(f"Rate limited. Waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None

    print("Max retries exceeded.")
    return None


def find_nearest_station(lat, lon, start_date, end_date, api_key, bbox_deg=0.5):
    """Find nearest global-hourly station using bounding box search."""
    extent = f"{lat - bbox_deg},{lon - bbox_deg},{lat + bbox_deg},{lon + bbox_deg}"

    params = {
        "datasetid": "GHCND",  # Use GHCND to find stations, then check for global-hourly
        "extent": extent,
        "startdate": start_date[:10],
        "enddate": end_date[:10],
        "limit": 100
    }

    result = noaa_request("stations", params, api_key)

    if not result or "results" not in result:
        print("No stations found in bounding box. Expanding search...")
        params["extent"] = f"{lat - 1},{lon - 1},{lat + 1},{lon + 1}"
        result = noaa_request("stations", params, api_key)

    if not result or "results" not in result:
        return None

    stations = result["results"]

    # Calculate distances and sort
    for station in stations:
        slat = station.get("latitude", 0)
        slon = station.get("longitude", 0)
        station["distance"] = np.sqrt((slat - lat)**2 + (slon - lon)**2)

    stations.sort(key=lambda x: x["distance"])

    # Return the nearest station
    nearest = stations[0]
    print(f"Found station: {nearest['id']} - {nearest.get('name', 'Unknown')}")
    print(f"  Distance: ~{nearest['distance'] * 111:.1f} km")
    return nearest["id"]


def fetch_noaa_data(station_id, start_date, end_date, api_key):
    """Fetch global-hourly data with pagination."""
    all_results = []
    offset = 1
    limit = 1000

    while True:
        params = {
            "datasetid": "global-hourly",
            "stationid": station_id,
            "startdate": start_date,
            "enddate": end_date,
            "limit": limit,
            "offset": offset,
            "units": "metric"
        }

        result = noaa_request("data", params, api_key)

        if not result or "results" not in result:
            break

        batch = result["results"]
        all_results.extend(batch)
        print(f"  Fetched {len(all_results)} records...")

        if len(batch) < limit:
            break

        offset += limit
        time.sleep(0.3)  # Rate limit courtesy

    return all_results

## Data Parsing Functions

In [ ]:
def parse_global_hourly_value(value_str):
    """Parse global-hourly encoded values (format: value,quality_code)."""
    if pd.isna(value_str) or value_str in ["", "9999", "+9999"]:
        return np.nan

    try:
        # Handle comma-separated format
        if "," in str(value_str):
            parts = str(value_str).split(",")
            val = parts[0].strip()
        else:
            val = str(value_str).strip()

        # Remove leading + sign
        val = val.lstrip("+")

        if val in ["", "9999", "99999", "999"]:
            return np.nan

        return float(val)
    except:
        return np.nan


def calculate_rh_from_dewpoint(temp_c, dewpoint_c):
    """Calculate relative humidity from temperature and dewpoint (Magnus formula)."""
    if pd.isna(temp_c) or pd.isna(dewpoint_c):
        return np.nan

    a = 17.27
    b = 237.7

    gamma_dp = (a * dewpoint_c) / (b + dewpoint_c)
    gamma_t = (a * temp_c) / (b + temp_c)

    rh = 100 * np.exp(gamma_dp - gamma_t)
    return min(100, max(0, rh))


def process_noaa_records(records):
    """Process raw NOAA records into a structured DataFrame."""
    if not records:
        return pd.DataFrame()

    # Group by timestamp
    grouped = {}
    for rec in records:
        ts = rec.get("date", "")
        if ts not in grouped:
            grouped[ts] = {}

        datatype = rec.get("datatype", "")
        value = rec.get("value", "")
        grouped[ts][datatype] = value

    rows = []
    for ts, data in grouped.items():
        row = {"timestamp_utc": ts}

        # Temperature (TMP is in tenths of degree C)
        tmp = parse_global_hourly_value(data.get("TMP"))
        row["ambient_temp_C"] = tmp / 10 if not pd.isna(tmp) else np.nan

        # Dewpoint (DEW is in tenths of degree C)
        dew = parse_global_hourly_value(data.get("DEW"))
        dewpoint_c = dew / 10 if not pd.isna(dew) else np.nan

        # Relative humidity - use direct if available, else compute
        rh = parse_global_hourly_value(data.get("RH"))
        if pd.isna(rh) and not pd.isna(row["ambient_temp_C"]) and not pd.isna(dewpoint_c):
            rh = calculate_rh_from_dewpoint(row["ambient_temp_C"], dewpoint_c)
        row["relative_humidity_pct"] = rh

        # Wind speed (WND format: direction,direction_quality,type,speed,speed_quality)
        wnd = data.get("WND", "")
        if wnd and "," in str(wnd):
            parts = str(wnd).split(",")
            if len(parts) >= 4:
                wind_dir = parse_global_hourly_value(parts[0])
                wind_spd = parse_global_hourly_value(parts[3])
                row["wind_dir_deg"] = wind_dir if wind_dir != 999 else np.nan
                row["wind_speed_mps"] = wind_spd / 10 if not pd.isna(wind_spd) else np.nan
            else:
                row["wind_dir_deg"] = np.nan
                row["wind_speed_mps"] = np.nan
        else:
            row["wind_dir_deg"] = np.nan
            row["wind_speed_mps"] = np.nan

        # Sky cover (raw value)
        sky = data.get("GF1", data.get("SKY", ""))
        row["sky_cover_raw"] = str(sky) if sky else np.nan

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
        df = df.sort_values("timestamp_utc").reset_index(drop=True)

    return df

## Fetch NOAA Data

In [ ]:
# Validate API key
if NOAA_API_KEY == "e97d47ec-be0b-11ee-84d9-0242ac130002-e97d48aa-be0b-11ee-84d9-0242ac130002" or not NOAA_API_KEY:
    raise ValueError("Please set your NOAA_API_KEY in the configuration cell above.")

# Find or use provided station
if STATION_ID:
    station_id = STATION_ID
    print(f"Using provided station: {station_id}")
else:
    print("Auto-selecting nearest station...")
    station_id = find_nearest_station(LAT, LON, START_UTC, END_UTC, NOAA_API_KEY)

    if not station_id:
        raise ValueError("Could not find a nearby station. Please provide STATION_ID manually.")

# Fetch data
print(f"\nFetching global-hourly data from {START_UTC} to {END_UTC}...")
raw_records = fetch_noaa_data(station_id, START_UTC, END_UTC, NOAA_API_KEY)

if not raw_records:
    print("\nNo data returned. This could mean:")
    print("  - The station doesn't have global-hourly data for this period")
    print("  - The time range is too recent (data may not be available yet)")
    print("  - Try a different STATION_ID or expand the time range")
else:
    print(f"\nTotal records fetched: {len(raw_records)}")

ValueError: Please set your NOAA_API_KEY in the configuration cell above.

## Process and Add Solar Position

In [ ]:
# Process records
df_noaa = process_noaa_records(raw_records)

if df_noaa.empty:
    print("Warning: No valid data after processing. Creating empty template.")
    df_noaa = pd.DataFrame(columns=[
        "timestamp_utc", "ambient_temp_C", "relative_humidity_pct",
        "wind_speed_mps", "wind_dir_deg", "sky_cover_raw",
        "solar_elevation_deg", "solar_azimuth_deg"
    ])
else:
    # Calculate solar position using pvlib
    print("Calculating solar positions...")

    solar_pos = get_solarposition(
        time=df_noaa["timestamp_utc"],
        latitude=LAT,
        longitude=LON
    )

    df_noaa["solar_elevation_deg"] = solar_pos["elevation"].values
    df_noaa["solar_azimuth_deg"] = solar_pos["azimuth"].values

    print(f"\nProcessed {len(df_noaa)} hourly records.")
    print(f"Date range: {df_noaa['timestamp_utc'].min()} to {df_noaa['timestamp_utc'].max()}")

    # Show data availability
    print("\nData availability:")
    for col in df_noaa.columns:
        if col != "timestamp_utc":
            valid = df_noaa[col].notna().sum()
            print(f"  {col}: {valid}/{len(df_noaa)} ({100*valid/len(df_noaa):.1f}%)")

# Save to CSV
noaa_output_path = f"{OUTPUT_DIR}/noaa_env_hourly.csv"
df_noaa.to_csv(noaa_output_path, index=False)
print(f"\nSaved: {noaa_output_path}")

# Display sample
df_noaa.head(10)

---
# Phase B: Thermal Image Analysis & Correlation

## Upload Images

Upload your thermal images (PNG, JPG, TIF) to `/content/thermal/`.

In [ ]:
from google.colab import files

print(f"Upload thermal images to: {THERMAL_DIR}")
print("Supported formats: PNG, JPG, JPEG, TIF, TIFF")
print("\nNote: IRX files will be detected but require a custom parser.")
print("")

uploaded = files.upload()

# Move uploaded files to thermal directory
for filename in uploaded.keys():
    src = f"/content/{filename}"
    dst = f"{THERMAL_DIR}/{filename}"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"Moved: {filename} -> {THERMAL_DIR}/")

# List contents
print(f"\nFiles in {THERMAL_DIR}:")
for f in os.listdir(THERMAL_DIR):
    print(f"  {f}")

## Optional: Upload Timestamp Mapping CSV

If your filenames don't contain timestamps, upload a CSV with columns: `filename,timestamp_utc`

In [ ]:
# Optional: Upload a timestamp mapping file
TIMESTAMP_MAPPING_PATH = None  # Will be set if uploaded

print("Optional: Upload a CSV mapping file with columns: filename,timestamp_utc")
print("Skip this cell (Ctrl+Enter without uploading) if not needed.")
print("")

try:
    mapping_upload = files.upload()
    if mapping_upload:
        mapping_filename = list(mapping_upload.keys())[0]
        TIMESTAMP_MAPPING_PATH = f"/content/{mapping_filename}"
        print(f"\nMapping file uploaded: {TIMESTAMP_MAPPING_PATH}")
except:
    print("No mapping file uploaded. Will use EXIF/filename parsing.")

## Image Processing Functions

In [ ]:
# Timestamp extraction regex pattern
# Matches: YYYYMMDD_HHMMSS, YYYY-MM-DD_HH-MM-SS, YYYYMMDDHHMMSS, etc.
TIMESTAMP_REGEX = r"(\d{4})[_-]?(\d{2})[_-]?(\d{2})[_-]?(\d{2})[_-]?(\d{2})[_-]?(\d{2})"


def get_exif_datetime(image_path):
    """Extract DateTimeOriginal from EXIF data."""
    try:
        img = Image.open(image_path)
        exif_data = img._getexif()

        if exif_data:
            for tag_id, value in exif_data.items():
                tag = TAGS.get(tag_id, tag_id)
                if tag == "DateTimeOriginal":
                    # Format: "YYYY:MM:DD HH:MM:SS"
                    return datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
    except:
        pass
    return None


def parse_filename_timestamp(filename):
    """Extract timestamp from filename using regex.

    Pattern matches:
    - YYYYMMDD_HHMMSS (e.g., 20240101_143022)
    - YYYY-MM-DD_HH-MM-SS (e.g., 2024-01-01_14-30-22)
    - YYYYMMDDHHMMSS (e.g., 20240101143022)
    """
    match = re.search(TIMESTAMP_REGEX, filename)
    if match:
        year, month, day, hour, minute, second = match.groups()
        try:
            return datetime(int(year), int(month), int(day),
                          int(hour), int(minute), int(second))
        except:
            pass
    return None


def extract_image_timestamp(image_path, mapping_df=None):
    """Extract timestamp using priority: mapping CSV > EXIF > filename regex."""
    filename = os.path.basename(image_path)

    # 1. Check mapping CSV first
    if mapping_df is not None and not mapping_df.empty:
        match = mapping_df[mapping_df["filename"] == filename]
        if not match.empty:
            return pd.to_datetime(match.iloc[0]["timestamp_utc"]).to_pydatetime()

    # 2. Try EXIF DateTimeOriginal
    exif_dt = get_exif_datetime(image_path)
    if exif_dt:
        return exif_dt

    # 3. Try filename regex
    filename_dt = parse_filename_timestamp(filename)
    if filename_dt:
        return filename_dt

    return None


def compute_intensity_metrics(image_path):
    """Load image as grayscale and compute intensity statistics."""
    try:
        img = Image.open(image_path).convert("L")  # Convert to grayscale
        arr = np.array(img, dtype=np.float64)

        return {
            "mean_intensity": np.mean(arr),
            "median_intensity": np.median(arr),
            "p90_intensity": np.percentile(arr, 90),
            "std_intensity": np.std(arr)
        }
    except Exception as e:
        print(f"  Error processing {image_path}: {e}")
        return None

## Process Images

In [ ]:
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
IRX_EXTENSION = ".irx"

# Load mapping CSV if provided
mapping_df = None
if TIMESTAMP_MAPPING_PATH and os.path.exists(TIMESTAMP_MAPPING_PATH):
    mapping_df = pd.read_csv(TIMESTAMP_MAPPING_PATH)
    print(f"Loaded timestamp mapping with {len(mapping_df)} entries.")

# Get image files
image_files = []
irx_files = []

for f in os.listdir(THERMAL_DIR):
    ext = os.path.splitext(f)[1].lower()
    filepath = os.path.join(THERMAL_DIR, f)

    if ext in SUPPORTED_EXTENSIONS:
        image_files.append(filepath)
    elif ext == IRX_EXTENSION:
        irx_files.append(f)

# Warn about IRX files
if irx_files:
    print("=" * 60)
    print("WARNING: IRX files detected but cannot be processed automatically.")
    print("IRX is a proprietary format requiring a custom parser.")
    print(f"Skipped files: {', '.join(irx_files)}")
    print("Please convert these to PNG/TIF using the manufacturer's software.")
    print("=" * 60)
    print()

if not image_files:
    print("No supported image files found in /content/thermal/")
    print("Please upload PNG, JPG, or TIF images.")
else:
    print(f"Processing {len(image_files)} images...\n")

# Process each image
image_data = []
no_timestamp = []

for img_path in sorted(image_files):
    filename = os.path.basename(img_path)
    print(f"Processing: {filename}")

    # Extract timestamp
    timestamp = extract_image_timestamp(img_path, mapping_df)

    if timestamp is None:
        print(f"  WARNING: Could not extract timestamp. Skipping.")
        no_timestamp.append(filename)
        continue

    print(f"  Timestamp: {timestamp}")

    # Compute intensity metrics
    metrics = compute_intensity_metrics(img_path)

    if metrics is None:
        continue

    image_data.append({
        "filename": filename,
        "timestamp_utc": timestamp,
        **metrics
    })

    print(f"  Mean: {metrics['mean_intensity']:.2f}, Std: {metrics['std_intensity']:.2f}")

# Create DataFrame
df_images = pd.DataFrame(image_data)

if not df_images.empty:
    df_images["timestamp_utc"] = pd.to_datetime(df_images["timestamp_utc"])

print(f"\nProcessed {len(df_images)} images successfully.")

if no_timestamp:
    print(f"\nImages without timestamps (skipped): {len(no_timestamp)}")
    for f in no_timestamp:
        print(f"  - {f}")
    print("\nTo include these, either:")
    print(f"  1. Rename files to include timestamp (pattern: {TIMESTAMP_REGEX})")
    print("  2. Upload a mapping CSV with columns: filename,timestamp_utc")

## Merge with NOAA Data

In [ ]:
MAX_TIME_DELTA_MINUTES = 30

if df_images.empty:
    print("No images to merge. Please process images first.")
    df_merged = pd.DataFrame()
elif df_noaa.empty:
    print("No NOAA data to merge with. Please fetch NOAA data first.")
    df_merged = pd.DataFrame()
else:
    print(f"Merging images with NOAA data (max {MAX_TIME_DELTA_MINUTES} min tolerance)...\n")

    # Use merge_asof for nearest timestamp matching
    df_images_sorted = df_images.sort_values("timestamp_utc")
    df_noaa_sorted = df_noaa.sort_values("timestamp_utc")

    df_merged = pd.merge_asof(
        df_images_sorted,
        df_noaa_sorted,
        on="timestamp_utc",
        direction="nearest",
        tolerance=pd.Timedelta(minutes=MAX_TIME_DELTA_MINUTES),
        suffixes=("", "_noaa")
    )

    # Check for unmatched (NaN in NOAA columns)
    unmatched_mask = df_merged["ambient_temp_C"].isna()
    unmatched = df_merged[unmatched_mask]
    matched = df_merged[~unmatched_mask]

    if not unmatched.empty:
        print(f"Unmatched images (no NOAA data within {MAX_TIME_DELTA_MINUTES} min):")
        for _, row in unmatched.iterrows():
            print(f"  - {row['filename']} at {row['timestamp_utc']}")
        print()

    # Keep only matched rows
    df_merged = matched.reset_index(drop=True)

    print(f"Successfully merged: {len(df_merged)} images")
    print(f"Dropped (unmatched): {len(unmatched)} images")

# Save merged data
merged_output_path = f"{OUTPUT_DIR}/image_env_merged.csv"

if not df_merged.empty:
    df_merged.to_csv(merged_output_path, index=False)
    print(f"\nSaved: {merged_output_path}")
    display(df_merged)
else:
    print("\nNo merged data to save.")

## Correlation Analysis

In [ ]:
if df_merged.empty or len(df_merged) < 3:
    print("Insufficient data for correlation analysis (need at least 3 matched images).")
else:
    # Define variables
    image_metrics = ["mean_intensity", "median_intensity", "p90_intensity", "std_intensity"]
    env_vars = ["ambient_temp_C", "relative_humidity_pct", "wind_speed_mps",
                "wind_dir_deg", "solar_elevation_deg", "solar_azimuth_deg"]

    # Filter to variables with data
    env_vars_available = [v for v in env_vars if v in df_merged.columns and df_merged[v].notna().sum() >= 3]

    print("Computing correlations...\n")

    # Calculate correlations
    correlation_results = []

    for img_metric in image_metrics:
        for env_var in env_vars_available:
            # Get valid pairs
            valid_mask = df_merged[img_metric].notna() & df_merged[env_var].notna()

            if valid_mask.sum() < 3:
                continue

            x = df_merged.loc[valid_mask, env_var]
            y = df_merged.loc[valid_mask, img_metric]

            # Pearson correlation
            pearson_r, pearson_p = stats.pearsonr(x, y)

            # Spearman correlation
            spearman_r, spearman_p = stats.spearmanr(x, y)

            correlation_results.append({
                "image_metric": img_metric,
                "env_variable": env_var,
                "n_samples": valid_mask.sum(),
                "pearson_r": pearson_r,
                "pearson_p": pearson_p,
                "spearman_r": spearman_r,
                "spearman_p": spearman_p,
                "abs_pearson_r": abs(pearson_r)
            })

    df_corr = pd.DataFrame(correlation_results)

    # Save correlations
    corr_output_path = f"{OUTPUT_DIR}/correlation_table.csv"
    df_corr.to_csv(corr_output_path, index=False)
    print(f"Saved: {corr_output_path}\n")

    # Print ranked correlations
    print("=" * 60)
    print("STRONGEST CORRELATIONS (ranked by |Pearson r|)")
    print("=" * 60)

    df_corr_sorted = df_corr.sort_values("abs_pearson_r", ascending=False)

    for i, row in df_corr_sorted.iterrows():
        sig_pearson = "*" if row["pearson_p"] < 0.05 else ""
        sig_spearman = "*" if row["spearman_p"] < 0.05 else ""

        print(f"{row['image_metric']:20} vs {row['env_variable']:25} | "
              f"Pearson: {row['pearson_r']:+.3f}{sig_pearson:1} | "
              f"Spearman: {row['spearman_r']:+.3f}{sig_spearman:1} | "
              f"n={row['n_samples']}")

    print("\n* indicates p < 0.05")

## OLS Regression

In [ ]:
if df_merged.empty or len(df_merged) < 5:
    print("Insufficient data for regression (need at least 5 matched images).")
else:
    print("OLS Regression: mean_intensity ~ environmental variables\n")

    # Define predictors
    predictors = ["ambient_temp_C", "wind_speed_mps", "solar_elevation_deg", "relative_humidity_pct"]

    # Check for sky_cover encoding
    if "sky_cover_raw" in df_merged.columns and df_merged["sky_cover_raw"].notna().any():
        # Simple encoding: convert to numeric if possible, else create dummy
        try:
            df_merged["sky_cover_encoded"] = pd.to_numeric(
                df_merged["sky_cover_raw"].astype(str).str.extract(r"(\d+)")[0],
                errors="coerce"
            )
            if df_merged["sky_cover_encoded"].notna().sum() > 0:
                predictors.append("sky_cover_encoded")
                print("Note: sky_cover_raw encoded as numeric.")
        except:
            pass

    # Filter to available predictors with data
    available_predictors = [p for p in predictors
                           if p in df_merged.columns and df_merged[p].notna().sum() > 0]

    if not available_predictors:
        print("No valid predictors available for regression.")
    else:
        # Prepare data - drop rows with any NaN in relevant columns
        cols_needed = ["mean_intensity"] + available_predictors
        df_reg = df_merged[cols_needed].dropna()

        if len(df_reg) < 5:
            print(f"Only {len(df_reg)} complete cases. Need at least 5 for regression.")
        else:
            print(f"Using {len(df_reg)} complete observations.")
            print(f"Predictors: {available_predictors}\n")

            # Prepare X and y
            X = df_reg[available_predictors]
            X = sm.add_constant(X)  # Add intercept
            y = df_reg["mean_intensity"]

            # Fit OLS
            model = sm.OLS(y, X).fit()

            # Print summary
            print(model.summary())

## Visualization

In [ ]:
if df_merged.empty:
    print("No data to plot.")
else:
    # Check if we have the required columns
    if "solar_elevation_deg" not in df_merged.columns:
        print("solar_elevation_deg not available in merged data.")
    else:
        # Filter valid data
        plot_data = df_merged[["mean_intensity", "solar_elevation_deg"]].dropna()

        if plot_data.empty:
            print("No valid data points for scatter plot.")
        else:
            fig, ax = plt.subplots(figsize=(8, 6))

            ax.scatter(
                plot_data["solar_elevation_deg"],
                plot_data["mean_intensity"],
                alpha=0.7,
                edgecolors="black",
                linewidths=0.5,
                s=60
            )

            # Add trend line if enough points
            if len(plot_data) >= 3:
                z = np.polyfit(plot_data["solar_elevation_deg"], plot_data["mean_intensity"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(plot_data["solar_elevation_deg"].min(),
                                    plot_data["solar_elevation_deg"].max(), 100)
                ax.plot(x_line, p(x_line), "r--", alpha=0.8, label="Linear fit")

                # Compute correlation for label
                r, p_val = stats.pearsonr(plot_data["solar_elevation_deg"], plot_data["mean_intensity"])
                ax.text(0.05, 0.95, f"r = {r:.3f}\np = {p_val:.3f}",
                       transform=ax.transAxes, verticalalignment="top",
                       bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

            ax.set_xlabel("Solar Elevation (degrees)", fontsize=12)
            ax.set_ylabel("Mean Intensity (grayscale)", fontsize=12)
            ax.set_title(f"Thermal Image Intensity vs Solar Elevation\n{SITE_LABEL}", fontsize=14)
            ax.grid(True, alpha=0.3)

            if len(plot_data) >= 3:
                ax.legend()

            plt.tight_layout()

            # Save figure
            fig_path = f"{OUTPUT_DIR}/scatter_intensity_vs_solar.png"
            plt.savefig(fig_path, dpi=150, bbox_inches="tight")
            print(f"Saved: {fig_path}")

            plt.show()

---
## Summary of Outputs

All outputs are saved to `/content/`:

In [ ]:
print("Generated files:")
print("=" * 50)

output_files = [
    ("noaa_env_hourly.csv", "NOAA environmental data with solar position"),
    ("image_env_merged.csv", "Images merged with environmental data"),
    ("correlation_table.csv", "Pearson & Spearman correlations"),
    ("scatter_intensity_vs_solar.png", "Scatter plot visualization")
]

for filename, description in output_files:
    filepath = f"{OUTPUT_DIR}/{filename}"
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"✓ {filename:35} ({size:,} bytes) - {description}")
    else:
        print(f"✗ {filename:35} (not created) - {description}")